# Knowledge Distillation Experiment: Image Classification (MNIST)

## Experiment Goals:
1. Train a large CNN teacher on MNIST
2. Train a small CNN student using distillation
3. Compare with a control student of the same architecture trained without the teacher
4. Analyze performance vs model size tradeoff

The protocol follows Hinton, Vinyals and Dean (2015).
The teacher is trained first and then frozen.
The distilled student and the control student share the architecture, the optimizer, the number of steps, the initial weights and the order of the batches, so the only difference between them is the loss.
A validation split held out from the training set is used while training, and the test set is read once, at the end.

## Setup and Dependencies

In [ ]:
import os
# hide tensorflow's c++ start up messages so the saved outputs only show the experiment
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

# python side tensorflow notices, such as the one about gpus on windows, would otherwise land in the outputs
tf.get_logger().setLevel('ERROR')

print(f"tensorflow version: {tf.__version__}")
print(f"keras version: {keras.__version__}")

In [ ]:
SEED = 0
# fixed seeds alone do not make tensorflow repeatable, deterministic kernels are needed as well
tf.config.experimental.enable_op_determinism()

## Load MNIST Dataset

In [ ]:
# mnist: 60k train and 10k test 28x28 grayscale digits, 10 classes
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

# scale to [0, 1] and add the channel axis the conv layers expect
x_train_full = (x_train_full.astype('float32') / 255.0)[..., None]
x_test = (x_test.astype('float32') / 255.0)[..., None]

Hold out 6,000 training images as a validation set.
Everything that is looked at during training uses it, so the test set stays untouched until the final comparison.

In [ ]:
perm = np.random.default_rng(SEED).permutation(len(x_train_full))
val_idx, train_idx = perm[:6000], perm[6000:]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

In [ ]:
print(f"training samples: {len(x_train)}")
print(f"validation samples: {len(x_val)}")
print(f"test samples: {len(x_test)}")
print(f"image shape: {x_train.shape[1:]}")

a few training digits with their labels

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f'label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Define Teacher and Student Models

Both models end in a plain `Dense(10)` layer, so they output logits.
Distillation softens logits with `softmax(z / T)`.
Applying it to outputs that already went through a softmax would soften probabilities instead.
A probability vector divided by `T = 4` sums to 0.25, so after the second softmax every class lands between 0.097 and 0.125 (computed in the next cell), which says almost nothing about the teacher.

In [ ]:
# p / 4 sums to 0.25, both extremes come from a one hot p: the top class gets e^0.25 / (e^0.25 + 9)
# and every other class 1 / (e^0.25 + 9)
e = np.exp(0.25)
print(f"softmax(p / 4) for any probability vector p: every class between {1 / (e + 9):.3f} and {e / (e + 9):.3f}")

In [ ]:
# teacher: a large cnn, only its accuracy matters because it is never deployed
def create_teacher_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(256, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu'),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.5),
        # no softmax: the model emits logits so the distillation loss can soften them with softmax(z / t)
        keras.layers.Dense(10),
    ])
    return model

In [ ]:
# student: about 8x fewer parameters than the teacher, the model meant for deployment
def create_student_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.3),
        # no softmax, as in the teacher
        keras.layers.Dense(10),
    ])
    return model

In [ ]:
create_teacher_model().summary()

In [ ]:
create_student_model().summary()

In [ ]:
teacher_params = create_teacher_model().count_params()
student_params = create_student_model().count_params()

print("model architecture comparison:")
print(f"teacher parameters: {teacher_params:,}")
print(f"student parameters: {student_params:,}")
print(f"compression ratio: {teacher_params/student_params:.1f}x")
print(f"size reduction: {(1-student_params/teacher_params)*100:.1f}%")

## Training Functions

The loss is `alpha * soft + (1 - alpha) * hard`.
The soft term is `KL(softmax(teacher / T) || softmax(student / T))` multiplied by `T^2`, the hard term is the ordinary cross entropy with the labels at `T = 1`.
With `alpha = 0` the loss is plain supervised training, which is how the teacher and the control student are trained.

In [ ]:
# distillation loss after hinton, vinyals and dean (2015): alpha weights the soft term, 1 - alpha the hard term
def distillation_loss(y_true, student_logits, teacher_logits, temperature, alpha):
    # hard term at t=1: cross entropy with the true labels keeps the student anchored to the ground truth
    hard = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=tf.cast(y_true, tf.int32), logits=student_logits))
    if alpha == 0:
        return hard
    # soft term: kl(teacher_t || student_t) on logits, a temperature above 1 exposes how the teacher ranks
    # the wrong classes, log space avoids an epsilon, times t^2 because soft target gradients shrink as 1 / t^2
    teacher_log = tf.nn.log_softmax(teacher_logits / temperature)
    student_log = tf.nn.log_softmax(student_logits / temperature)
    soft = tf.reduce_mean(tf.reduce_sum(tf.exp(teacher_log) * (teacher_log - student_log), axis=1)) * temperature ** 2
    return alpha * soft + (1 - alpha) * hard

One training step, compiled with `tf.function`.
A custom step is needed because `fit()` has no hook for feeding the teacher's outputs into the loss.

In [ ]:
def make_train_step(model, optimizer, temperature, alpha):
    @tf.function(reduce_retracing=True)
    def train_step(x, y, teacher_logits):
        with tf.GradientTape() as tape:
            logits = model(x, training=True)
            loss = distillation_loss(y, logits, teacher_logits, temperature, alpha)
        # only this model's variables get gradients, which is what keeps the teacher frozen
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        return loss
    return train_step

In [ ]:
def accuracy(model, x, y):
    # inference mode turns dropout off, argmax of the logits is the same at every temperature
    logits = model.predict(x, batch_size=1000, verbose=0)
    return float(np.mean(np.argmax(logits, axis=1) == y))

The same loop trains the teacher, the control student and the distilled student.
For a given seed the control and the distilled student start from the same weights and see the batches in the same order.

In [ ]:
def train(model, x, y, teacher_logits=None, temperature=1.0, alpha=0.0, epochs=10,
          batch_size=128, seed=0, eval_every=1):
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    train_step = make_train_step(model, optimizer, temperature, alpha)
    # one generator per run, so the batch order depends on the seed and nothing else
    rng = np.random.default_rng(seed)
    history = {'epoch': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(x))
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            batch_teacher = None if teacher_logits is None else teacher_logits[idx]
            train_step(x[idx], y[idx], batch_teacher)
        if epoch % eval_every == 0 or epoch == epochs:
            history['epoch'].append(epoch)
            history['val_acc'].append(accuracy(model, x_val, y_val))
    return history

In [ ]:
def new_model(create, seed):
    # the global seed fixes the initial weights and the dropout masks of the model built next
    keras.utils.set_random_seed(seed)
    return create()

## Train Teacher Model

In [ ]:
# the teacher is trained first, on hard labels only
t0 = time.time()
teacher = new_model(create_teacher_model, SEED)
teacher_history = train(teacher, x_train, y_train, epochs=10, seed=SEED)
print(f"teacher trained in {time.time() - t0:.0f} s")
print(f"teacher validation accuracy by epoch: {np.round(teacher_history['val_acc'], 4).tolist()}")

Freeze the teacher and compute its logits for every training image once.
There is no data augmentation, so these are exactly the logits the teacher would produce batch by batch.

In [ ]:
teacher.trainable = False
teacher_train_logits = teacher.predict(x_train, batch_size=1000, verbose=0)
print(f"teacher logits: {teacher_train_logits.shape}")

## Train the Students

In [ ]:
temperature = 4.0
alpha = 0.7
epochs = 10

In [ ]:
t0 = time.time()
student_distilled = new_model(create_student_model, SEED)
distilled_history = train(student_distilled, x_train, y_train, teacher_logits=teacher_train_logits,
                          temperature=temperature, alpha=alpha, epochs=epochs, seed=SEED)
print(f"distilled student trained in {time.time() - t0:.0f} s")

In [ ]:
# control: same architecture, seed, steps and batch order, hard labels only
t0 = time.time()
student_baseline = new_model(create_student_model, SEED)
baseline_history = train(student_baseline, x_train, y_train, epochs=epochs, seed=SEED)
print(f"control student trained in {time.time() - t0:.0f} s")

## Results Comparison

Validation accuracy during training, all three models evaluated with dropout off.

In [ ]:
RESULTS_DIR = os.path.join('..', 'results', 'image_classification')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(teacher_history['epoch'], np.array(teacher_history['val_acc']) * 100, marker='o', label='teacher')
ax.plot(distilled_history['epoch'], np.array(distilled_history['val_acc']) * 100, marker='s', label='student (distilled)')
ax.plot(baseline_history['epoch'], np.array(baseline_history['val_acc']) * 100, marker='^', label='student (control)')
ax.set_title('validation accuracy during training', fontweight='bold')
ax.set_xlabel('epoch')
ax.set_ylabel('accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'distillation_results.png'), dpi=150, bbox_inches='tight')
plt.show()

The test set is read here for the first time.

In [ ]:
teacher_acc = accuracy(teacher, x_test, y_test) * 100
distilled_acc = accuracy(student_distilled, x_test, y_test) * 100
baseline_acc = accuracy(student_baseline, x_test, y_test) * 100

In [ ]:
print("test accuracy:")
print(f"teacher (large model): {teacher_acc:.2f}%")
print(f"student with distillation: {distilled_acc:.2f}%")
print(f"student without distillation: {baseline_acc:.2f}%")

In [ ]:
print(f"distilled minus control: {distilled_acc - baseline_acc:+.2f} percentage points")
print(f"gap to teacher (with distillation): {teacher_acc - distilled_acc:.2f} percentage points")
print(f"gap to teacher (control): {teacher_acc - baseline_acc:.2f} percentage points")

In [ ]:
print(f"model size reduction: {(1-student_params/teacher_params)*100:.1f}%")
print(f"performance retained: {(distilled_acc/teacher_acc)*100:.1f}%")

## Inference Speed Comparison

In [ ]:
# wall clock latency per batch, the deployment cost a smaller student is meant to cut
def measure_inference_ms(model, x, batch_size=128, repeats=50):
    forward = tf.function(lambda batch: model(batch, training=False))
    batch = tf.constant(x[:batch_size])
    # the first call traces the graph, keep it out of the timing
    forward(batch)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        forward(batch).numpy()
        times.append(time.perf_counter() - start)
    # the median is robust to the occasional scheduler hiccup
    return float(np.median(times) * 1000)

In [ ]:
teacher_time = measure_inference_ms(teacher, x_test)
student_time = measure_inference_ms(student_distilled, x_test)

In [ ]:
print("inference speed comparison (median over 50 batches of 128):")
print(f"teacher inference time: {teacher_time:.2f} ms/batch")
print(f"student inference time: {student_time:.2f} ms/batch")
print(f"speedup: {teacher_time/student_time:.2f}x")

## Visualize Predictions

In [ ]:
test_images = x_test[:5]
test_labels = y_test[:5]

teacher_preds = np.argmax(teacher.predict(test_images, verbose=0), axis=1)
distilled_preds = np.argmax(student_distilled.predict(test_images, verbose=0), axis=1)
baseline_preds = np.argmax(student_baseline.predict(test_images, verbose=0), axis=1)

In [ ]:
rows = [('teacher', teacher_preds), ('distilled', distilled_preds), ('control', baseline_preds)]
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row, (name, preds) in enumerate(rows):
    for i in range(5):
        ax = axes[row, i]
        ax.imshow(test_images[i].squeeze(), cmap='gray')
        # hide ticks but keep the axis, axis('off') would also hide the row label
        ax.set_xticks([])
        ax.set_yticks([])
        color = 'green' if preds[i] == test_labels[i] else 'red'
        ax.set_title(f'{name}: {preds[i]}\ntrue: {test_labels[i]}', color=color, fontweight='bold')
    axes[row, 0].set_ylabel(name, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'prediction_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()